# Exploración de datos de Instacart con Python y Pandas

**Evidencia:** GA1-220501094-01-AA1-EV04 – Fundamentos en Pandas. Explorando Instacart: descubre patrones de compra

**Aprendiz:** Maicol Esneider Posada Callejas

**Programa:** Tecnólogo en Análisis y Desarrollo de Software (228118) · **Ficha:** 3223873

**Instructor:** Carlos Navia · Servicio Nacional de Aprendizaje (SENA)

**Fecha:** octubre de 2026

---

**Contenido**

1. Comprensión del problema y reconocimiento de los datos
2. Importación y exploración inicial de los archivos
3. Reconocimiento de la estructura y calidad inicial de los datos
4. Indexación y filtrado lógico de la información
5. Obtención e interpretación de indicadores básicos
6. Resolución del problema y conclusiones

> **Cómo ejecutar este notebook.** Los cinco archivos CSV deben estar en la carpeta `data/raw/` del proyecto y el notebook se abre desde `notebooks/`. Las instrucciones completas están en el `README.md`. Después se usa *Run → Run All Cells*.

## Momento 1. Comprensión del problema y reconocimiento de los datos

### Descripción del problema

Instacart es una plataforma de compras de supermercado por internet: el cliente arma su pedido en la aplicación y se lo llevan a la casa. Cada vez que alguien compra, la plataforma guarda qué pidió, a qué hora, qué día y cuántos días pasaron desde su compra anterior.

En esta actividad asumo el rol de analista de datos de Instacart. El equipo de operaciones tiene cinco archivos con pedidos, productos, pasillos, departamentos y los artículos de cada pedido, pero todavía no sabe bien qué información contiene cada uno ni cómo consultarla. Mi tarea es hacer una **exploración inicial**: conocer la estructura de los archivos, revisar su calidad, consultar registros específicos y calcular indicadores básicos sobre los hábitos de compra.

**Pregunta orientadora:** ¿Cómo utilizar Python y la librería Pandas para explorar los datos de Instacart, identificar las características de los pedidos y productos, y obtener información relevante sobre los hábitos de compra de sus clientes?

### Los cinco archivos y lo que considero que almacenan

| N.º | Archivo | Qué considero que almacena | Una fila representa |
|---|---|---|---|
| 1 | `instacart_orders.csv` | Los pedidos: quién los hizo, qué día de la semana, a qué hora y cuántos días pasaron desde el pedido anterior del mismo cliente. | Un pedido |
| 2 | `products.csv` | El catálogo de productos con su nombre y el pasillo y departamento al que pertenecen. | Un producto |
| 3 | `aisles.csv` | La lista de pasillos (categorías pequeñas, por ejemplo *fresh fruits* o *yogurt*). | Un pasillo |
| 4 | `departments.csv` | La lista de departamentos (categorías grandes, por ejemplo *frozen* o *produce*). | Un departamento |
| 5 | `order_products.csv` | Qué productos iban dentro de cada pedido, en qué posición se agregaron al carrito y si el cliente ya los había comprado antes. | Un artículo dentro de un pedido |

### Columnas que identifican pedidos, usuarios y productos

- **Pedidos:** `order_id`. Aparece en `instacart_orders.csv` y en `order_products.csv`, así que sirve para unir un pedido con los productos que lleva.
- **Usuarios (clientes):** `user_id`, en `instacart_orders.csv`.
- **Productos:** `product_id`. Aparece en `products.csv` y en `order_products.csv`.
- Además, `aisle_id` une los productos con su pasillo y `department_id` los une con su departamento.

### Tres preguntas de análisis sobre los hábitos de compra

Las formulé para que se puedan responder con las operaciones básicas estudiadas (filtros, conteos, sumas y promedios):

1. **P1.** ¿En qué franja del día (mañana, tarde o noche) hacen más pedidos los clientes?
2. **P2.** ¿Qué proporción de los artículos comprados son productos que el cliente ya había pedido antes (recompras)?
3. **P3.** ¿Cada cuántos días vuelve a comprar un cliente en promedio, y cuántos pedidos se hacen más de 15 días después del pedido anterior?

Las respuestas están en el Momento 6.

### ¿Por qué es importante explorar los datos antes de analizarlos?

Porque no se puede sacar una conclusión confiable de datos que no se conocen. Antes de calcular cualquier indicador hay que saber qué representa cada fila, qué significa cada columna, de qué tipo es cada dato y si hay celdas vacías o registros repetidos. Por ejemplo, si no reviso que una fila de `instacart_orders.csv` es un pedido y no un cliente, podría decir que Instacart tiene casi 479.000 clientes cuando en realidad son muchos menos. La exploración también muestra los problemas de calidad (valores ausentes, duplicados) que pueden cambiar los resultados si no se tienen en cuenta.

## Momento 2. Importación y exploración inicial de los archivos

Primero importo la librería **pandas** con el alias `pd`. Luego cargo cada archivo con `pd.read_csv()`. Los archivos usan **punto y coma (`;`)** para separar las columnas, por eso agrego el parámetro `sep=";"`. Sin ese parámetro, pandas metería toda la fila en una sola columna.

In [1]:
import pandas as pd

# Carpeta donde están los archivos CSV, relativa a la carpeta notebooks/
RUTA = "../data/raw/"

orders = pd.read_csv(RUTA + "instacart_orders.csv", sep=";")
products = pd.read_csv(RUTA + "products.csv", sep=";")
aisles = pd.read_csv(RUTA + "aisles.csv", sep=";")
departments = pd.read_csv(RUTA + "departments.csv", sep=";")
order_products = pd.read_csv(RUTA + "order_products.csv", sep=";")

# Mostrar todas las columnas al imprimir una tabla
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Versión de pandas:", pd.__version__)
print("Los cinco archivos se cargaron correctamente.")

Versión de pandas: 3.0.5
Los cinco archivos se cargaron correctamente.


Cada archivo quedó guardado en un **DataFrame** (una tabla de pandas) con un nombre que dice qué contiene: `orders`, `products`, `aisles`, `departments` y `order_products`.

Ahora reviso cada uno con `head()` (primeras 5 filas), `tail()` (últimas 5 filas) y `shape` (cantidad de filas y columnas).

### Pedidos (`orders`)

In [2]:
print("Dimensiones (filas, columnas):", orders.shape)
print("Primeros 5 registros:")
display(orders.head())
print("Últimos 5 registros:")
display(orders.tail())

Dimensiones (filas, columnas): (478967, 6)
Primeros 5 registros:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
1,1690866,163593,5,5,12,9.0
2,1454967,39980,4,5,19,2.0
3,1768857,82516,56,0,20,10.0
4,3007858,196724,2,4,12,17.0


Últimos 5 registros:


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
478962,3210681,5617,5,1,14,7.0
478963,3270802,112087,2,3,13,6.0
478964,885349,82944,16,2,11,6.0
478965,216274,4391,3,3,8,8.0
478966,2071924,1730,18,1,14,15.0


### Productos (`products`)

In [3]:
print("Dimensiones (filas, columnas):", products.shape)
print("Primeros 5 registros:")
display(products.head())
print("Últimos 5 registros:")
display(products.tail())

Dimensiones (filas, columnas): (49694, 4)
Primeros 5 registros:


,product_id,product_name,aisle_id,department_id
0,1,Chocolate Sandwich Cookies,61,19
1,2,All-Seasons Salt,104,13
2,3,Robust Golden Unsweetened Oolong Tea,94,7
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
4,5,Green Chile Anytime Sauce,5,13


Últimos 5 registros:


,product_id,product_name,aisle_id,department_id
49689,49690,HIGH PERFORMANCE ENERGY DRINK,64,7
49690,49691,ORIGINAL PANCAKE & WAFFLE MIX,130,14
49691,49692,ORGANIC INSTANT OATMEAL LIGHT MAPLE BROWN SUGAR,130,14
49692,49693,SPRING WATER BODY WASH,127,11
49693,49694,BURRITO- STEAK & CHEESE,38,1


### Pasillos (`aisles`)

In [4]:
print("Dimensiones (filas, columnas):", aisles.shape)
print("Primeros 5 registros:")
display(aisles.head())
print("Últimos 5 registros:")
display(aisles.tail())

Dimensiones (filas, columnas): (134, 2)
Primeros 5 registros:


,aisle_id,aisle
0,1,prepared soups salads
1,2,specialty cheeses
2,3,energy granola bars
3,4,instant foods
4,5,marinades meat preparation


Últimos 5 registros:


,aisle_id,aisle
129,130,hot cereal pancake mixes
130,131,dry pasta
131,132,beauty
132,133,muscles joints pain relief
133,134,specialty wines champagnes


### Departamentos (`departments`)

In [5]:
print("Dimensiones (filas, columnas):", departments.shape)
print("Primeros 5 registros:")
display(departments.head())
print("Últimos 5 registros:")
display(departments.tail())

Dimensiones (filas, columnas): (21, 2)
Primeros 5 registros:


,department_id,department
0,1,frozen
1,2,other
2,3,bakery
3,4,produce
4,5,alcohol


Últimos 5 registros:


,department_id,department
16,17,household
17,18,babies
18,19,snacks
19,20,deli
20,21,missing


### Artículos de cada pedido (`order_products`)

In [6]:
print("Dimensiones (filas, columnas):", order_products.shape)
print("Primeros 5 registros:")
display(order_products.head())
print("Últimos 5 registros:")
display(order_products.tail())

Dimensiones (filas, columnas): (4545007, 4)
Primeros 5 registros:


,order_id,product_id,add_to_cart_order,reordered
0,2141543,11440,17.0,0
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1


Últimos 5 registros:


,order_id,product_id,add_to_cart_order,reordered
4545002,577211,15290,12.0,1
4545003,1219554,21914,9.0,0
4545004,692640,47766,4.0,1
4545005,319435,691,8.0,1
4545006,1398151,28733,9.0,0


### Resumen de dimensiones

Junto las dimensiones de los cinco DataFrames en una sola tabla para compararlas. `shape[0]` es el número de filas y `shape[1]` el número de columnas.

In [7]:
dimensiones = pd.DataFrame({
    "DataFrame": ["orders", "products", "aisles", "departments", "order_products"],
    "Filas": [orders.shape[0], products.shape[0], aisles.shape[0], departments.shape[0], order_products.shape[0]],
    "Columnas": [orders.shape[1], products.shape[1], aisles.shape[1], departments.shape[1], order_products.shape[1]],
})
dimensiones

,DataFrame,Filas,Columnas
0,orders,478967,6
1,products,49694,4
2,aisles,134,2
3,departments,21,2
4,order_products,4545007,4


Las dimensiones coinciden con las que indica la guía de aprendizaje para los cinco archivos.

### Respuestas a las preguntas del Momento 2

**¿Cuál de los cinco DataFrames contiene más registros?**
`order_products`, con **4.545.007 filas**. Es lógico: cada pedido trae varios productos y cada producto ocupa una fila.

**¿Cuál contiene menos columnas?**
`aisles` y `departments` empatan con **2 columnas** cada uno: un identificador y un nombre. Son tablas de referencia que solo traducen un número a un nombre.

**¿Qué diferencias identifico entre los datos de pedidos y los datos de productos?**
- `orders` registra **eventos**: cada fila es una compra que alguien hizo en un momento (día y hora). Tiene 478.967 filas y todas sus columnas son números.
- `products` es un **catálogo**: cada fila es un artículo que se puede comprar, haya sido comprado o no. Tiene 49.694 filas e incluye una columna de texto (`product_name`).
- `orders` responde *quién* compró y *cuándo*; `products` responde *qué* se vende y *en qué categoría* está. Se conectan a través de `order_products`.

**¿Por qué es importante conocer las dimensiones de un conjunto de datos?**
Porque dicen cuánta información hay y qué tan pesado será trabajar con ella (`order_products` tiene más de 4,5 millones de filas y tarda más en procesarse). También sirven de punto de comparación: si después de un filtro quedan muchas o pocas filas, se puede medir contra el total. Y ayudan a detectar errores de carga, por ejemplo, si un archivo aparece con 1 sola columna es porque se leyó con el separador equivocado.

## Momento 3. Reconocimiento de la estructura y calidad inicial de los datos

En este momento uso los atributos `columns`, `dtypes` y `shape`, y el método `info()`, para conocer cómo está construida cada tabla. **No corrijo ni elimino datos**: solo identifico sus características y posibles problemas.

### 3.1 Nombres de las columnas (`columns`)

In [8]:
print("orders:        ", list(orders.columns))
print("products:      ", list(products.columns))
print("aisles:        ", list(aisles.columns))
print("departments:   ", list(departments.columns))
print("order_products:", list(order_products.columns))

orders:         ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day', 'days_since_prior_order']
products:       ['product_id', 'product_name', 'aisle_id', 'department_id']
aisles:         ['aisle_id', 'aisle']
departments:    ['department_id', 'department']
order_products: ['order_id', 'product_id', 'add_to_cart_order', 'reordered']


### 3.2 Tipos de datos de cada columna (`dtypes`)

In [9]:
print("Tipos de datos de orders:")
print(orders.dtypes)

Tipos de datos de orders:
order_id                    int64
user_id                     int64
order_number                int64
order_dow                   int64
order_hour_of_day           int64
days_since_prior_order    float64
dtype: object


In [10]:
print("Tipos de datos de products:")
print(products.dtypes)

Tipos de datos de products:
product_id       int64
product_name       str
aisle_id         int64
department_id    int64
dtype: object


In [11]:
print("Tipos de datos de aisles:")
print(aisles.dtypes)

Tipos de datos de aisles:
aisle_id    int64
aisle         str
dtype: object


In [12]:
print("Tipos de datos de departments:")
print(departments.dtypes)

Tipos de datos de departments:
department_id    int64
department         str
dtype: object


In [13]:
print("Tipos de datos de order_products:")
print(order_products.dtypes)

Tipos de datos de order_products:
order_id               int64
product_id             int64
add_to_cart_order    float64
reordered              int64
dtype: object


**Clasificación de las columnas según su tipo:**

| Tipo | Qué guarda | Columnas |
|---|---|---|
| `int64` (entero) | Números sin decimales | `order_id`, `user_id`, `order_number`, `order_dow`, `order_hour_of_day`, `product_id`, `aisle_id`, `department_id`, `reordered` |
| `float64` (decimal) | Números con decimales | `days_since_prior_order`, `add_to_cart_order` |
| texto (`object` o `str`) | Palabras | `product_name`, `aisle`, `department` |

**Nota sobre la versión de pandas.** En pandas 3 (la versión que usé), las columnas de texto aparecen como `str`. En versiones anteriores, como la que trae Google Colab, aparecen como `object`. En los dos casos significa lo mismo: la columna guarda texto.

**Algo curioso:** `days_since_prior_order` y `add_to_cart_order` deberían ser números enteros (días completos y posiciones en el carrito), pero aparecen como `float64`. Esto pasa porque tienen **celdas vacías**: pandas marca una celda vacía con `NaN`, y `NaN` solo existe en el tipo decimal, así que convierte toda la columna a `float64`. Es la primera pista de que hay valores ausentes.

### 3.3 Campos que relacionan los conjuntos de datos

Estos campos aparecen en más de una tabla y permiten conectar la información:

| Campo | Tablas donde aparece | Qué relación permite |
|---|---|---|
| `order_id` | `orders` y `order_products` | Saber qué productos lleva cada pedido, y de qué cliente, día y hora es ese pedido |
| `product_id` | `products` y `order_products` | Conocer el nombre, pasillo y departamento de cada artículo comprado |
| `aisle_id` | `products` y `aisles` | Traducir el número de pasillo a su nombre |
| `department_id` | `products` y `departments` | Traducir el número de departamento a su nombre |

```
orders ──(order_id)── order_products ──(product_id)── products ──(aisle_id)────── aisles
                                                               └─(department_id)── departments
```

`user_id` solo está en `orders`. Para saber qué productos compró un cliente hay que pasar por `order_id`.

### 3.4 Información general con `info()`

`info()` muestra en una sola salida el número de filas, el nombre de cada columna, cuántos valores **no nulos** (celdas con dato) tiene cada una, su tipo y la memoria que ocupa.

In [14]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 478967 entries, 0 to 478966
Data columns (total 6 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   order_id                478967 non-null  int64  
 1   user_id                 478967 non-null  int64  
 2   order_number            478967 non-null  int64  
 3   order_dow               478967 non-null  int64  
 4   order_hour_of_day       478967 non-null  int64  
 5   days_since_prior_order  450148 non-null  float64
dtypes: float64(1), int64(5)
memory usage: 21.9 MB


In [15]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 49694 entries, 0 to 49693
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49694 non-null  int64
 1   product_name   48436 non-null  str  
 2   aisle_id       49694 non-null  int64
 3   department_id  49694 non-null  int64
dtypes: int64(3), str(1)
memory usage: 2.9 MB


In [16]:
aisles.info()

<class 'pandas.DataFrame'>
RangeIndex: 134 entries, 0 to 133
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   aisle_id  134 non-null    int64
 1   aisle     134 non-null    str  
dtypes: int64(1), str(1)
memory usage: 4.2 KB


In [17]:
departments.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   department_id  21 non-null     int64
 1   department     21 non-null     str  
dtypes: int64(1), str(1)
memory usage: 638.0 bytes


`order_products` es muy grande. Con tantas filas, `info()` deja de contar los no nulos por defecto, así que agrego `show_counts=True` para obligarlo a mostrarlos.

In [18]:
order_products.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 4545007 entries, 0 to 4545006
Data columns (total 4 columns):
 #   Column             Non-Null Count    Dtype  
---  ------             --------------    -----  
 0   order_id           4545007 non-null  int64  
 1   product_id         4545007 non-null  int64  
 2   add_to_cart_order  4544171 non-null  float64
 3   reordered          4545007 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 138.7 MB


### 3.5 Comparación entre registros y valores no nulos

Si una columna tiene menos valores no nulos que el total de filas, la diferencia son los **valores ausentes**. Para no hacer la resta a mano uso `isnull().sum()`, que cuenta las celdas vacías de cada columna.

In [19]:
print("Valores ausentes en orders:")
print(orders.isnull().sum(), "\n")
print("Valores ausentes en products:")
print(products.isnull().sum(), "\n")
print("Valores ausentes en aisles:")
print(aisles.isnull().sum(), "\n")
print("Valores ausentes en departments:")
print(departments.isnull().sum(), "\n")
print("Valores ausentes en order_products:")
print(order_products.isnull().sum())

Valores ausentes en orders:
order_id                      0
user_id                       0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order    28819
dtype: int64 

Valores ausentes en products:
product_id          0
product_name     1258
aisle_id            0
department_id       0
dtype: int64 

Valores ausentes en aisles:
aisle_id    0
aisle       0
dtype: int64 

Valores ausentes en departments:
department_id    0
department       0
dtype: int64 

Valores ausentes en order_products:
order_id               0
product_id             0
add_to_cart_order    836
reordered              0
dtype: int64


Hay valores ausentes en tres columnas:

- `orders['days_since_prior_order']`: **28.819** vacíos (478.967 filas − 450.148 no nulos).
- `products['product_name']`: **1.258** productos sin nombre.
- `order_products['add_to_cart_order']`: **836** vacíos.

Reviso un poco más cada caso para entender de dónde salen (sin corregir nada todavía).

In [20]:
# ¿Los vacíos de days_since_prior_order son de primeros pedidos?
primeros_pedidos = orders[orders["order_number"] == 1]
print("Pedidos con order_number = 1:", primeros_pedidos.shape[0])
print("Vacíos en days_since_prior_order:", orders["days_since_prior_order"].isnull().sum())
print("Vacíos que son primeros pedidos:", primeros_pedidos["days_since_prior_order"].isnull().sum())

Pedidos con order_number = 1: 28819
Vacíos en days_since_prior_order: 28819
Vacíos que son primeros pedidos: 28819


Los 28.819 vacíos coinciden exactamente con los 28.819 pedidos que son el **primer pedido** de un cliente (`order_number = 1`). Tiene sentido: si es la primera compra, no hay un pedido anterior desde el cual contar días. Estos vacíos no son un error, sino una falta de dato esperada.

In [21]:
# ¿A qué pasillo y departamento pertenecen los productos sin nombre?
productos_sin_nombre = products[products["product_name"].isnull()]
print("Pasillos de los productos sin nombre:", productos_sin_nombre["aisle_id"].unique())
print("Departamentos de los productos sin nombre:", productos_sin_nombre["department_id"].unique())
print(aisles[aisles["aisle_id"] == 100])
print(departments[departments["department_id"] == 21])

Pasillos de los productos sin nombre: [100]
Departamentos de los productos sin nombre: [21]
    aisle_id    aisle
99       100  missing
    department_id department
20             21    missing


Todos los productos sin nombre están en el pasillo 100 y el departamento 21, que se llaman literalmente **"missing"** (faltante). Es decir, son productos de los que no se conoce ni el nombre ni la categoría.

In [22]:
# ¿En cuántos pedidos aparecen los vacíos de add_to_cart_order?
pedidos_con_vacios = order_products[order_products["add_to_cart_order"].isnull()]["order_id"]
print("Pedidos distintos con algún vacío en add_to_cart_order:", pedidos_con_vacios.nunique())
print("Posición máxima registrada en el carrito:", order_products["add_to_cart_order"].max())

Pedidos distintos con algún vacío en add_to_cart_order: 70


Posición máxima registrada en el carrito: 64.0


Los 836 vacíos están concentrados en solo **70 pedidos**, y la posición más alta registrada en el carrito es 64. Esto sugiere que el sistema solo guardó la posición hasta el puesto 64 y dejó vacía la de los artículos que se agregaron después en pedidos muy grandes.

### 3.6 Registros duplicados

La descripción del caso dice que el instructor introdujo duplicados a propósito. Con `duplicated().sum()` cuento las filas que son copia exacta de otra.

In [23]:
print("Filas duplicadas en orders:", orders.duplicated().sum())
print("Filas duplicadas en products:", products.duplicated().sum())
print("Filas duplicadas en aisles:", aisles.duplicated().sum())
print("Filas duplicadas en departments:", departments.duplicated().sum())
print("Filas duplicadas en order_products:", order_products.duplicated().sum())

Filas duplicadas en orders: 15


Filas duplicadas en products: 0
Filas duplicadas en aisles: 0
Filas duplicadas en departments: 0


Filas duplicadas en order_products: 0


In [24]:
# Las filas repetidas de orders, ordenadas para ver cada par junto
orders[orders.duplicated(keep=False)].sort_values("order_id").head(6)

,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
354993,391768,57671,19,3,2,10.0
371905,391768,57671,19,3,2,10.0
119251,408114,68324,4,3,2,18.0
321100,408114,68324,4,3,2,18.0
394347,467134,63189,21,3,2,2.0
250626,467134,63189,21,3,2,2.0


In [25]:
# ¿En qué día y hora están las copias?
copias = orders[orders.duplicated()]
print("Días (order_dow) de las copias:", copias["order_dow"].unique())
print("Horas (order_hour_of_day) de las copias:", copias["order_hour_of_day"].unique())

Días (order_dow) de las copias: [3]
Horas (order_hour_of_day) de las copias: [2]


`orders` tiene **15 filas duplicadas**: el mismo pedido (mismo `order_id`, cliente, día y hora) aparece dos veces. Todas las copias son del miércoles (`order_dow = 3`) a las 2 a.m., lo que hace pensar en un error de registro en ese momento. Las demás tablas no tienen filas duplicadas.

### 3.7 Tabla de diagnóstico

In [26]:
tablas = {
    "instacart_orders.csv": orders,
    "products.csv": products,
    "aisles.csv": aisles,
    "departments.csv": departments,
    "order_products.csv": order_products,
}

filas, columnas, ausentes, duplicados = [], [], [], []
for nombre, df in tablas.items():
    filas.append(df.shape[0])
    columnas.append(df.shape[1])
    ausentes.append(df.isnull().sum().sum())
    duplicados.append(df.duplicated().sum())

diagnostico = pd.DataFrame({
    "Archivo": list(tablas.keys()),
    "Filas": filas,
    "Columnas": columnas,
    "Valores ausentes": ausentes,
    "Filas duplicadas": duplicados,
})
diagnostico

,Archivo,Filas,Columnas,Valores ausentes,Filas duplicadas
0,instacart_orders.csv,478967,6,28819,15
1,products.csv,49694,4,1258,0
2,aisles.csv,134,2,0,0
3,departments.csv,21,2,0,0
4,order_products.csv,4545007,4,836,0


| Archivo | Filas | Columnas | Observaciones sobre los datos |
|---|---|---|---|
| `instacart_orders.csv` | 478.967 | 6 | 28.819 vacíos en `days_since_prior_order` (son los primeros pedidos de cada cliente). 15 filas duplicadas, todas del miércoles a las 2 a.m. Hay 478.952 pedidos distintos y 157.437 clientes distintos. |
| `products.csv` | 49.694 | 4 | 1.258 productos sin nombre, todos en el pasillo y departamento "missing". Sin duplicados. |
| `aisles.csv` | 134 | 2 | Completo. El pasillo 100 se llama "missing". |
| `departments.csv` | 21 | 2 | Completo. El departamento 21 se llama "missing". |
| `order_products.csv` | 4.545.007 | 4 | 836 vacíos en `add_to_cart_order`, concentrados en 70 pedidos muy grandes (la posición máxima es 64). Es el archivo más pesado. Sin duplicados. |

### Respuestas a las preguntas del Momento 3

**¿Qué diferencias existen entre `int64`, `float64` y `object`?**
- `int64` guarda **números enteros**, sin decimales (por ejemplo, la hora 13 o el ID 1515936).
- `float64` guarda **números decimales** (por ejemplo, 9.0 o 11.5). También es el tipo que pandas usa cuando una columna numérica tiene celdas vacías (`NaN`).
- `object` guarda **texto** u otros datos que no son números (por ejemplo, "Chocolate Sandwich Cookies"). En pandas 3 el texto aparece como `str`.

Con los números se pueden hacer cuentas como promedios o sumas; con el texto no.

**¿Qué información proporciona `info()` que no se obtiene directamente con `shape`?**
`shape` solo dice cuántas filas y columnas hay. `info()` además muestra el nombre de cada columna, su tipo de dato, **cuántos valores no nulos tiene cada columna** (lo que permite detectar vacíos) y cuánta memoria ocupa la tabla.

**¿En qué columnas se observan valores ausentes?**
En `days_since_prior_order` (28.819), `product_name` (1.258) y `add_to_cart_order` (836).

**¿Qué posibles dificultades podrían generar estos valores en un análisis posterior?**
- Los promedios y conteos se calculan **solo con los valores disponibles**. Por ejemplo, el promedio de días entre pedidos deja por fuera los 28.819 primeros pedidos, y hay que decirlo al interpretar el resultado.
- Si se reemplazaran los vacíos por 0 sin pensar, se cambiaría el significado: un 0 en `days_since_prior_order` querría decir "volvió a comprar el mismo día", lo cual es falso.
- Los productos sin nombre no se pueden identificar en un reporte, por ejemplo en una lista de "productos más vendidos".
- Los vacíos obligan a que algunas columnas enteras queden como decimales, y eso puede confundir.
- Las filas duplicadas inflan los conteos: si cuento filas en lugar de pedidos distintos, obtengo 15 pedidos de más.

## Momento 4. Indexación y filtrado lógico de la información

- **Seleccionar columnas** es quedarme con algunas columnas de la tabla, pero con todas sus filas.
- **Filtrar filas** es quedarme con las filas que cumplen una condición, por ejemplo "la hora es igual a 10".

Para los dos casos uso `loc[]`. Su forma es `df.loc[filas, columnas]`: dentro de los corchetes indico qué filas quiero (o una condición) y qué columnas quiero. En cada ejercicio muestro el código, cuántos registros quedaron y las primeras filas del resultado.

### Ejercicio 1. Seleccionar únicamente `product_id` y `product_name` (products)

In [27]:
productos_id_nombre = products.loc[:, ["product_id", "product_name"]]

print("Dimensiones:", productos_id_nombre.shape)
productos_id_nombre.head()

Dimensiones: (49694, 2)


,product_id,product_name
0,1,Chocolate Sandwich Cookies
1,2,All-Seasons Salt
2,3,Robust Golden Unsweetened Oolong Tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...
4,5,Green Chile Anytime Sauce


**Explicación:** el `:` antes de la coma significa "todas las filas", y la lista después de la coma indica las dos columnas que quiero. El resultado conserva los 49.694 productos, pero ahora con 2 columnas en lugar de 4. Esto es **seleccionar columnas**: no se pierde ninguna fila.

### Ejercicio 2. Productos cuyo `department_id` es igual a 1 (products)

In [28]:
productos_depto_1 = products.loc[products["department_id"] == 1]

print("Productos del departamento 1:", productos_depto_1.shape[0])
print("Nombre del departamento 1:", departments.loc[departments["department_id"] == 1, "department"].values[0])
productos_depto_1.head()

Productos del departamento 1: 4008
Nombre del departamento 1: frozen


,product_id,product_name,aisle_id,department_id
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
29,30,Three Cheese Ziti- Marinara with Meatballs,38,1


**Explicación:** la condición `products["department_id"] == 1` revisa fila por fila si el departamento es 1 y devuelve `True` o `False`. `loc[]` solo deja pasar las filas con `True`. Quedan **4.008 productos** del departamento 1, que corresponde a **frozen** (congelados). Son el 8,1 % del catálogo.

### Ejercicio 3. Pedidos realizados a las 10 horas (orders)

In [29]:
pedidos_10h = orders.loc[orders["order_hour_of_day"] == 10]

print("Pedidos realizados a las 10:00:", pedidos_10h.shape[0])
pedidos_10h.head()

Pedidos realizados a las 10:00: 40578


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
28,133707,182261,1,3,10,NaN
36,3378084,127629,21,1,10,20.0
49,2161799,87345,7,0,10,24.0
51,1739741,118405,48,2,10,3.0


**Explicación:** hay **40.578 pedidos** hechos a las 10 de la mañana, el 8,5 % de todos los registros de pedidos. Es la hora con más pedidos de todo el día (lo verifico en el Momento 6, contando los pedidos de cada hora).

### Ejercicio 4. Pedidos cuyo `order_dow` es igual a 0 (orders)

In [30]:
pedidos_domingo = orders.loc[orders["order_dow"] == 0]

print("Pedidos realizados el domingo (order_dow = 0):", pedidos_domingo.shape[0])
pedidos_domingo.head()

Pedidos realizados el domingo (order_dow = 0): 84090


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
3,1768857,82516,56,0,20,10.0
5,1843871,20084,5,0,11,15.0
7,2521283,97711,2,0,18,22.0
23,3406282,19356,21,0,0,14.0
25,1895419,169793,2,0,11,8.0


**Explicación:** según el diccionario de datos, `order_dow = 0` es **domingo**. Ese día se hicieron **84.090 pedidos**, el 17,6 % del total. Si los pedidos se repartieran igual en los 7 días, a cada día le tocaría cerca del 14,3 %, así que el domingo está por encima del promedio.

### Ejercicio 5. Pedidos con `order_number` superior a 10 (orders)

In [31]:
pedidos_numero_mayor_10 = orders.loc[orders["order_number"] > 10]

print("Pedidos con order_number mayor que 10:", pedidos_numero_mayor_10.shape[0])
pedidos_numero_mayor_10.head()

Pedidos con order_number mayor que 10: 244072


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
3,1768857,82516,56,0,20,10.0
6,2511640,148233,40,3,8,2.0
8,2266494,34954,40,5,15,4.0
10,1162566,43738,33,4,11,2.0


**Explicación:** `order_number` dice qué número de pedido es para ese cliente (1 = su primera compra, 2 = la segunda, etc.). Hay **244.072 pedidos** que son del pedido 11 en adelante, es decir, el 51,0 % de los registros. Casi la mitad de los pedidos vienen de clientes que ya habían comprado más de 10 veces.

### Ejercicio 6. Pedidos cuyo `days_since_prior_order` es superior a 15 (orders)

In [32]:
pedidos_mas_15_dias = orders.loc[orders["days_since_prior_order"] > 15]

print("Pedidos hechos más de 15 días después del anterior:", pedidos_mas_15_dias.shape[0])
pedidos_mas_15_dias.head()

Pedidos hechos más de 15 días después del anterior: 110706


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
0,1515936,183418,11,6,13,30.0
4,3007858,196724,2,4,12,17.0
7,2521283,97711,2,0,18,22.0
14,2843219,74699,4,6,11,30.0
26,1844247,122558,4,6,18,30.0


**Explicación:** quedan **110.706 pedidos** en los que el cliente tardó más de 15 días en volver a comprar. Los vacíos (`NaN`) de los primeros pedidos **no cumplen** la condición `> 15`, así que quedan por fuera automáticamente. Por eso este número se debe comparar con los 450.148 pedidos que sí tienen el dato: representa el 24,6 % de ellos.

### Ejercicio 7. Registros cuyo valor de `reordered` es igual a 1 (order_products)

In [33]:
articulos_recomprados = order_products.loc[order_products["reordered"] == 1]

print("Artículos que el cliente ya había comprado antes:", articulos_recomprados.shape[0])
articulos_recomprados.head()

Artículos que el cliente ya había comprado antes: 2683838


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
3,491251,8670,35.0,1
4,2571142,1940,5.0,1
5,2456893,21616,4.0,1


**Explicación:** `reordered = 1` significa que el cliente ya había pedido ese producto en una compra anterior. Hay **2.683.838 artículos** recomprados, el 59,1 % de todos los artículos. Más de la mitad de lo que se compra son productos repetidos.

### Ejercicio 8. Productos agregados en la primera posición del carrito (order_products)

In [34]:
primeros_en_carrito = order_products.loc[order_products["add_to_cart_order"] == 1]

print("Artículos agregados de primeros al carrito:", primeros_en_carrito.shape[0])
print("Pedidos distintos en order_products:", order_products["order_id"].nunique())
primeros_en_carrito.head()

Artículos agregados de primeros al carrito: 450046


Pedidos distintos en order_products: 450046


,order_id,product_id,add_to_cart_order,reordered
1,567889,1560,1.0,1
2,2261212,26683,1.0,1
14,1961225,37553,1.0,1
16,639939,10017,1.0,1
23,750040,8518,1.0,0


**Explicación:** hay **450.046 artículos** que se agregaron de primeros al carrito. Ese número es igual a la cantidad de pedidos distintos que hay en `order_products` (450.046), lo cual tiene sentido: **cada pedido tiene exactamente un primer producto**. Este filtro sirve, entonces, para quedarse con una fila por pedido.

### Filtros con condiciones simultáneas

Para pedir que se cumplan **dos condiciones al mismo tiempo** uso el operador `&` (que significa "y"). Cada condición va entre paréntesis.

**Filtro A.** Pedidos realizados a las 10 horas **y** con `order_number` superior a 10.

In [35]:
pedidos_10h_y_mas_10 = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]

print("Pedidos a las 10:00:", pedidos_10h.shape[0])
print("Pedidos con order_number > 10:", pedidos_numero_mayor_10.shape[0])
print("Pedidos que cumplen las dos condiciones:", pedidos_10h_y_mas_10.shape[0])
pedidos_10h_y_mas_10.head()

Pedidos a las 10:00: 40578
Pedidos con order_number > 10: 244072
Pedidos que cumplen las dos condiciones: 21537


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order
19,294269,107523,29,2,10,1.0
36,3378084,127629,21,1,10,20.0
51,1739741,118405,48,2,10,3.0
92,60724,110808,78,0,10,2.0
140,3156095,113920,14,2,10,2.0


**Explicación:** de los 40.578 pedidos de las 10:00, solo **21.537** son de clientes que ya llevaban más de 10 compras (53,1 %). Al sumar una segunda condición, la cantidad de registros **baja**, porque cada fila tiene que pasar los dos filtros.

El mismo resultado se puede obtener aplicando los filtros uno después del otro, como en el material de estudio:

In [36]:
paso_1 = orders[orders["order_hour_of_day"] == 10]
paso_2 = paso_1[paso_1["order_number"] > 10]
print("Filtrando paso a paso:", paso_2.shape[0])

Filtrando paso a paso: 21537


**Filtro B.** Productos que pertenecen al departamento 1 **y** tienen un `aisle_id` superior a 50.

In [37]:
productos_depto_1_pasillo_50 = products.loc[(products["department_id"] == 1) & (products["aisle_id"] > 50)]

print("Productos del departamento 1:", productos_depto_1.shape[0])
print("Productos del departamento 1 con aisle_id > 50:", productos_depto_1_pasillo_50.shape[0])
productos_depto_1_pasillo_50.head()

Productos del departamento 1: 4008
Productos del departamento 1 con aisle_id > 50: 1618


,product_id,product_name,aisle_id,department_id
7,8,Cut Russet Potatoes Steam N' Mash,116,1
11,12,Chocolate Fudge Layer Cake,119,1
17,18,Pizza for One Suprema Frozen Pizza,79,1
45,46,Raisin Cinnamon Bagels 5 count,58,1
127,128,Organic Wild Blueberries,116,1


**Explicación:** de los 4.008 productos congelados, **1.618** están en pasillos con número mayor que 50. Otra vez, la segunda condición reduce el resultado.

**¿Qué pasa si me equivoco de operador?** Si en el filtro A uso `|` ("o") en lugar de `&` ("y"), el resultado cambia por completo:

In [38]:
con_y = orders.loc[(orders["order_hour_of_day"] == 10) & (orders["order_number"] > 10)]
con_o = orders.loc[(orders["order_hour_of_day"] == 10) | (orders["order_number"] > 10)]

print("Con & (se cumplen las dos):     ", con_y.shape[0])
print("Con | (se cumple al menos una): ", con_o.shape[0])

Con & (se cumplen las dos):      21537
Con | (se cumple al menos una):  263113


Con `|` salen **263.113** pedidos en lugar de 21.537, porque basta con que se cumpla una de las dos condiciones. Si no me doy cuenta del error, respondería una pregunta distinta a la que me hicieron.

### Respuestas a las preguntas del Momento 4

**¿Qué diferencia existe entre seleccionar columnas y filtrar filas?**
Seleccionar columnas cambia el **ancho** de la tabla: me quedo con algunas columnas, pero conservo todas las filas (ejercicio 1: 49.694 filas, 2 columnas). Filtrar filas cambia el **largo**: conservo todas las columnas, pero solo las filas que cumplen una condición (ejercicio 3: de 478.967 a 40.578 filas).

**¿Qué ventajas ofrece `loc[]` al consultar información específica?**
Permite indicar filas y columnas en la misma instrucción (`df.loc[condición, columnas]`), acepta condiciones lógicas para filtrar, trabaja con los nombres de las columnas en lugar de posiciones (el código se entiende mejor) y deja claro qué se está seleccionando.

**¿Qué ocurre con la cantidad de registros cuando se aplican varias condiciones?**
Con `&` la cantidad **baja o, como mucho, se mantiene igual**, porque cada fila tiene que cumplir todas las condiciones. En el filtro A se pasó de 40.578 a 21.537 pedidos, y en el filtro B de 4.008 a 1.618 productos.

**¿Por qué es importante utilizar correctamente los operadores lógicos?**
Porque cada operador hace una pregunta diferente. Como se vio arriba, cambiar `&` por `|` hizo que el resultado pasara de 21.537 a 263.113 pedidos. También importa la forma de escribirlos en pandas: hay que usar `&` y `|` (no las palabras `and` y `or`) y poner cada condición entre paréntesis; si no, Python da error o evalúa en un orden distinto al que se quería.

## Momento 5. Obtención e interpretación de indicadores básicos

En este momento uso `shape`, `count()`, `mean()` (promedio), `sum()` (suma) y `describe()` (resumen estadístico).

**Importante:** una cosa es la cantidad de **registros** (filas), otra la cantidad de **pedidos** y otra la cantidad de **usuarios** (clientes). Para contar valores distintos uso `nunique()`, que cuenta cuántos valores diferentes hay en una columna.

### 5.1 ¿Cuántos registros de pedidos contiene el archivo?

In [39]:
print("Registros (filas) en orders:", orders.shape[0])
print("Registros contados con count():", orders["order_id"].count())
print("Pedidos distintos (order_id únicos):", orders["order_id"].nunique())
print("Usuarios distintos (user_id únicos):", orders["user_id"].nunique())
print("Pedidos por usuario en promedio:", round(orders["order_id"].nunique() / orders["user_id"].nunique(), 2))

Registros (filas) en orders: 478967
Registros contados con count(): 478967
Pedidos distintos (order_id únicos): 478952
Usuarios distintos (user_id únicos): 157437


Pedidos por usuario en promedio: 3.04


### 5.2 ¿Cuántos registros de productos contiene el catálogo?

In [40]:
print("Productos en el catálogo:", products.shape[0])
print("Productos distintos (product_id únicos):", products["product_id"].nunique())
print("Productos distintos que aparecen en algún pedido:", order_products["product_id"].nunique())

Productos en el catálogo: 49694
Productos distintos (product_id únicos): 49694
Productos distintos que aparecen en algún pedido: 45573


### 5.3 ¿Cuál es el promedio de `order_number`?

In [41]:
promedio_order_number = orders["order_number"].mean()
print("Promedio de order_number:", round(promedio_order_number, 2))
print("Mediana de order_number:", orders["order_number"].median())
print("Valor máximo de order_number:", orders["order_number"].max())

Promedio de order_number: 17.16
Mediana de order_number: 11.0
Valor máximo de order_number: 100


### 5.4 ¿Cuál es el promedio de `days_since_prior_order` entre los valores disponibles?

In [42]:
promedio_dias = orders["days_since_prior_order"].mean()
print("Valores disponibles (no nulos):", orders["days_since_prior_order"].count())
print("Promedio de días entre pedidos:", round(promedio_dias, 2))
print("Mediana de días entre pedidos:", orders["days_since_prior_order"].median())
print("Pedidos con exactamente 30 días (el valor máximo):", orders[orders["days_since_prior_order"] == 30].shape[0])

Valores disponibles (no nulos): 450148
Promedio de días entre pedidos: 11.1
Mediana de días entre pedidos: 7.0
Pedidos con exactamente 30 días (el valor máximo): 51338


### 5.5 ¿Cuántos registros de artículos tienen `reordered` igual a 1?

In [43]:
recompras = order_products[order_products["reordered"] == 1]["reordered"].count()
print("Artículos con reordered = 1:", recompras)
print("Porcentaje sobre el total de artículos:", round(recompras / order_products.shape[0] * 100, 2), "%")

Artículos con reordered = 1: 2683838
Porcentaje sobre el total de artículos: 59.05 %


### 5.6 ¿Cuál es la suma de la columna `reordered`?

In [44]:
suma_reordered = order_products["reordered"].sum()
print("Suma de reordered:", suma_reordered)
print("¿Es igual al conteo anterior?", suma_reordered == recompras)

Suma de reordered: 2683838
¿Es igual al conteo anterior? True


### 5.7 ¿Qué características estadísticas presenta `order_hour_of_day`?

In [45]:
orders["order_hour_of_day"].describe().round(2)

count    478967.00
mean         13.45
std           4.22
min           0.00
25%          10.00
50%          13.00
75%          16.00
max          23.00
Name: order_hour_of_day, dtype: float64

### 5.8 ¿Qué características estadísticas presenta `add_to_cart_order`?

In [46]:
order_products["add_to_cart_order"].describe().round(2)

count    4544171.00
mean           8.35
std            7.08
min            1.00
25%            3.00
50%            6.00
75%           11.00
max           64.00
Name: add_to_cart_order, dtype: float64

### Tabla de indicadores e interpretación

| Pregunta analizada | Valor obtenido | Interpretación |
|---|---|---|
| ¿Cuántos registros de pedidos contiene el archivo? | 478.967 registros | Son filas, no clientes. Hay **478.952 pedidos distintos** (las 15 filas de diferencia son duplicados) y **157.437 usuarios distintos**. En promedio, cada cliente aparece con unos 3,0 pedidos en esta muestra. |
| ¿Cuántos registros de productos contiene el catálogo? | 49.694 productos | Cada fila es un producto diferente. De ellos, 45.573 aparecen en al menos un pedido de la muestra (91,7 %). |
| ¿Cuál es el promedio de `order_number`? | 17,16 | En promedio, un pedido registrado es el pedido número 17 de su cliente. La mediana es 11: la mitad de los pedidos son del pedido 11 o anterior. Que el promedio sea mayor que la mediana indica que unos pocos clientes muy frecuentes (con hasta 100 pedidos) suben el promedio. **No significa** que cada cliente haga 17 pedidos. |
| ¿Cuál es el promedio de `days_since_prior_order` entre los valores disponibles? | 11,10 días | Calculado sobre 450.148 pedidos (se excluyen los 28.819 primeros pedidos, que no tienen dato). La mediana es 7 días: la mitad de los pedidos se hicieron una semana o menos después del pedido anterior. Hay 51.338 pedidos con exactamente 30 días, el valor máximo; puede que 30 signifique "30 días o más", y entonces el promedio real sería algo mayor. |
| ¿Cuántos registros de artículos tienen `reordered` igual a 1? | 2.683.838 artículos | El 59,05 % de los artículos comprados son productos que el cliente ya había pedido antes. Son artículos, no pedidos ni clientes. |
| ¿Cuál es la suma de la columna `reordered`? | 2.683.838 | Da lo mismo que el conteo anterior porque `reordered` solo vale 0 o 1: los ceros no suman nada y cada uno cuenta como 1. Sumar una columna de 0 y 1 es una forma rápida de contar los 1. |
| ¿Qué características estadísticas presenta `order_hour_of_day`? | media 13,45; mediana 13; mín. 0; máx. 23; Q1 = 10; Q3 = 16; desv. est. 4,22 | Los pedidos van de las 0 a las 23 horas. La mitad de los pedidos se hacen **entre las 10 a.m. y las 4 p.m.** (entre Q1 y Q3), y la hora central es la 1 p.m. La desviación de unas 4 horas indica que la actividad se concentra en el día y hay pocos pedidos de madrugada. |
| ¿Qué características estadísticas presenta `add_to_cart_order`? | conteo 4.544.171; media 8,35; mediana 6; mín. 1; máx. 64; Q3 = 11 | El conteo es menor que el total de filas porque faltan 836 valores. La mediana de 6 indica que la mitad de los artículos se agregan entre las primeras 6 posiciones del carrito. La media (8,35) es mayor por los pedidos muy grandes. El máximo de 64 parece un límite del registro, como se vio en el Momento 3. |

## Momento 6. Resolución del problema y presentación del producto

### 6.1 Respuestas a mis tres preguntas de análisis

**P1. ¿En qué franja del día (mañana, tarde o noche) hacen más pedidos los clientes?**

Uso la misma función `franja` del proyecto práctico del entorno: mañana es antes de las 12, tarde de las 12 a las 18 y noche desde las 19. Después cuento los pedidos de cada franja con filtros.

In [47]:
def franja(order_hour_of_day):
    """Clasifica la hora del pedido en mañana, tarde o noche."""
    if order_hour_of_day < 12:
        return "mañana"
    elif order_hour_of_day < 19:
        return "tarde"
    else:
        return "noche"

for hora in [8, 13, 21]:
    print(hora, "->", franja(hora))

8 -> mañana
13 -> tarde
21 -> noche


In [48]:
total_registros = orders.shape[0]

manana = orders[orders["order_hour_of_day"] < 12]["order_id"].count()
tarde = orders[(orders["order_hour_of_day"] >= 12) & (orders["order_hour_of_day"] < 19)]["order_id"].count()
noche = orders[orders["order_hour_of_day"] >= 19]["order_id"].count()

print("Mañana (0 a 11 h): ", manana, "->", round(manana / total_registros * 100, 1), "%")
print("Tarde  (12 a 18 h):", tarde, "->", round(tarde / total_registros * 100, 1), "%")
print("Noche  (19 a 23 h):", noche, "->", round(noche / total_registros * 100, 1), "%")
print("Suma de las tres franjas:", manana + tarde + noche, "| Total de registros:", total_registros)

Mañana (0 a 11 h):  167641 -> 35.0 %
Tarde  (12 a 18 h): 252013 -> 52.6 %
Noche  (19 a 23 h): 59313 -> 12.4 %
Suma de las tres franjas: 478967 | Total de registros: 478967


Para ver qué hora individual tiene más pedidos, cuento los pedidos de cada una de las 24 horas:

In [49]:
hora_maxima = 0
pedidos_hora_maxima = 0

for hora in range(24):
    cantidad = orders[orders["order_hour_of_day"] == hora]["order_id"].count()
    print(str(hora).rjust(2), "h:", cantidad)
    if cantidad > pedidos_hora_maxima:
        hora_maxima = hora
        pedidos_hora_maxima = cantidad

print("\nHora con más pedidos:", hora_maxima, "h, con", pedidos_hora_maxima, "pedidos")

 0 h: 3180
 1 h: 1763
 2 h: 1004
 3 h: 770
 4 h: 765
 5 h: 1371
 6 h: 4215
 7 h: 13043
 8 h: 25024
 9 h: 35896
10 h: 40578
11 h: 40032
12 h: 38034
13 h: 39007
14 h: 39631
15 h: 39789
16 h: 38112
17 h: 31930
18 h: 25510
19 h: 19547
20 h: 14624
21 h: 11019
22 h: 8512
23 h: 5611

Hora con más pedidos: 10 h, con 40578 pedidos


**Respuesta P1:** la **tarde** es la franja con más pedidos: 252.013 (52,6 %). Le sigue la mañana con 167.641 (35,0 %) y por último la noche con 59.313 (12,4 %). Aunque la hora con más pedidos es las 10 a.m., la tarde gana porque tiene muchas horas seguidas con mucha actividad (de 12 a 6 p.m.).

Para complementar, cuento también los pedidos por día de la semana con un ciclo `for` y un filtro por cada día:

In [50]:
nombres_dias = ["Domingo", "Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]

for numero_dia in range(7):
    cantidad = orders[orders["order_dow"] == numero_dia]["order_id"].count()
    print(numero_dia, nombres_dias[numero_dia].ljust(10), cantidad, "->", round(cantidad / total_registros * 100, 1), "%")

0 Domingo    84090 -> 17.6 %
1 Lunes      82185 -> 17.2 %
2 Martes     65833 -> 13.7 %
3 Miércoles  60912 -> 12.7 %
4 Jueves     59810 -> 12.5 %
5 Viernes    63488 -> 13.3 %
6 Sábado     62649 -> 13.1 %


El **domingo** (84.090 pedidos) y el **lunes** (82.185) son los días con más pedidos; de martes a sábado la cantidad baja y se mantiene parecida (entre 59.810 y 65.833).

**P2. ¿Qué proporción de los artículos comprados son productos que el cliente ya había pedido antes (recompras)?**

In [51]:
total_articulos = order_products.shape[0]
articulos_recompra = order_products["reordered"].sum()

print("Total de artículos comprados:", total_articulos)
print("Artículos de recompra:", articulos_recompra)
print("Proporción de recompras:", round(articulos_recompra / total_articulos * 100, 2), "%")
print("Promedio de la columna reordered:", round(order_products["reordered"].mean(), 4))

Total de artículos comprados: 4545007
Artículos de recompra: 2683838
Proporción de recompras: 59.05 %
Promedio de la columna reordered: 0.5905


**Respuesta P2:** el **59,05 %** de los artículos comprados (2.683.838 de 4.545.007) son productos que el cliente ya había pedido antes. Un dato útil: el promedio de una columna de 0 y 1 es justamente la proporción de unos, por eso `mean()` da 0,5905.

**P3. ¿Cada cuántos días vuelve a comprar un cliente en promedio, y cuántos pedidos se hacen más de 15 días después del pedido anterior?**

In [52]:
dias_disponibles = orders["days_since_prior_order"].count()
mas_de_15 = orders[orders["days_since_prior_order"] > 15]["order_id"].count()
hasta_7 = orders[orders["days_since_prior_order"] <= 7]["order_id"].count()

print("Promedio de días entre pedidos:", round(orders["days_since_prior_order"].mean(), 2))
print("Pedidos con dato de días:", dias_disponibles)
print("Pedidos hechos 7 días o menos después del anterior:", hasta_7, "->", round(hasta_7 / dias_disponibles * 100, 1), "%")
print("Pedidos hechos más de 15 días después del anterior:", mas_de_15, "->", round(mas_de_15 / dias_disponibles * 100, 1), "%")

Promedio de días entre pedidos: 11.1
Pedidos con dato de días: 450148
Pedidos hechos 7 días o menos después del anterior: 226747 -> 50.4 %
Pedidos hechos más de 15 días después del anterior: 110706 -> 24.6 %


**Respuesta P3:** en promedio, un cliente vuelve a comprar cada **11,1 días**. Pero la mayoría compra más seguido: el 50,4 % de los pedidos se hicieron 7 días o menos después del anterior. **110.706 pedidos** (24,6 % de los que tienen el dato) se hicieron más de 15 días después del pedido anterior. Este cálculo excluye los primeros pedidos, que no tienen un pedido anterior.

### 6.2 Principales hallazgos sobre los pedidos y productos

- **Volumen:** la muestra tiene 478.952 pedidos distintos de 157.437 clientes, con 4.545.007 artículos comprados de un catálogo de 49.694 productos.
- **Horario:** los clientes compran sobre todo de día. La mitad de los pedidos se concentra entre las 10 a.m. y las 4 p.m., la hora con más pedidos es las 10 a.m. (40.578) y la tarde es la franja más fuerte (52,6 %).
- **Días:** el domingo (17,6 %) y el lunes (17,2 %) son los días con más pedidos.
- **Fidelidad:** el 51,0 % de los pedidos son de clientes que ya llevaban más de 10 compras, y el 59,1 % de los artículos son recompras. Los clientes repiten mucho, tanto la plataforma como los productos.
- **Frecuencia:** la mitad de los pedidos se hicieron 7 días o menos después del anterior (mediana), aunque el promedio es de 11,1 días por los pedidos que llegan después de un mes.
- **Carrito:** la mitad de los artículos se agregan dentro de las primeras 6 posiciones; los pedidos pueden tener hasta 64 posiciones registradas.
- **Catálogo:** el departamento 1 (*frozen*) tiene 4.008 productos, el 8,1 % del catálogo.

### 6.3 Cómo la indexación y los filtros permitieron consultar información relevante

Sin filtros solo se tendrían tablas enormes, imposibles de revisar a ojo (`order_products` tiene más de 4,5 millones de filas). Con la indexación por columnas (`loc[:, [...]]`) me quedé solo con la información necesaria, como el ID y el nombre de los productos. Con el filtrado lógico convertí preguntas de negocio en condiciones: "¿cuántos pedidos hay a las 10?" se volvió `orders["order_hour_of_day"] == 10`, y "¿cuántos artículos son recompras?" se volvió `order_products["reordered"] == 1`. Al combinar condiciones con `&` pude hacer preguntas más precisas, como los pedidos de clientes frecuentes hechos a las 10 a.m. (21.537). Finalmente, aplicar `count()`, `sum()` o `mean()` sobre las tablas filtradas convirtió cada consulta en un número concreto, que es lo que permitió responder las tres preguntas.

### 6.4 Limitaciones identificadas en los conjuntos de datos

1. **Valores ausentes:** 28.819 en `days_since_prior_order` (primeros pedidos), 1.258 productos sin nombre ni categoría ("missing") y 836 posiciones del carrito vacías.
2. **Duplicados:** 15 pedidos repetidos en `orders`, todos del miércoles a las 2 a.m. Contar filas en lugar de pedidos distintos da un resultado inflado.
3. **Posibles topes en los valores:** `days_since_prior_order` llega máximo a 30 (51.338 pedidos tienen ese valor) y `add_to_cart_order` máximo a 64. Es posible que 30 signifique "30 o más días" y que no se guarden posiciones después de la 64, lo cual afecta los promedios.
4. **Muestra modificada:** el instructor redujo el conjunto original e introdujo errores a propósito, así que los números describen esta muestra y no toda la operación real de Instacart.
5. **Falta de contexto:** no hay fechas exactas (solo día de la semana y hora), ni precios, ni cantidades por artículo, ni ubicación de los clientes. Tampoco se sabe en qué zona horaria está la hora. Son datos de 2017 de Estados Unidos, por lo que no se pueden trasladar directamente a otro país.
6. **Solo exploración:** en esta fase no se limpiaron los datos, y las relaciones entre tablas se describieron pero no se unieron (por ejemplo, no se calculó qué departamento vende más).

### 6.5 Conclusiones

1. **Los clientes de Instacart son principalmente recurrentes.** El 59,1 % de los artículos comprados son recompras y casi la mitad de los pedidos (51,0 %) vienen de clientes con más de 10 compras previas. Además, la mitad de los pedidos se hacen 7 días o menos después del anterior, lo que indica que muchos clientes hacen su mercado más o menos cada semana. Esto sugiere que funciones como "volver a pedir" o listas de productos frecuentes serían muy útiles para la plataforma.

2. **La demanda se concentra en el día y al inicio de la semana.** La tarde reúne el 52,6 % de los pedidos, la mitad de las compras ocurre entre las 10 a.m. y las 4 p.m., y el domingo y el lunes son los días más activos. Con esta información, la empresa podría programar más personal y repartidores en esos horarios y días.

3. **Explorar la calidad de los datos antes de calcular cambia la interpretación.** Si no hubiera revisado la estructura, habría confundido 478.967 filas con clientes (en realidad son 157.437), habría contado 15 pedidos duplicados de más y habría pensado que los vacíos de `days_since_prior_order` eran errores, cuando son los primeros pedidos. Pandas permitió detectar todo esto con pocas instrucciones (`shape`, `info()`, `isnull()`, `duplicated()`, `nunique()`), y por eso la exploración inicial es un paso obligatorio antes de cualquier análisis.

### Respuesta a la pregunta orientadora

Python y Pandas permiten explorar los datos de Instacart siguiendo un orden: **cargar** los archivos con `read_csv()` usando el separador correcto; **conocer su estructura** con `head()`, `tail()`, `shape`, `columns`, `dtypes` e `info()`; **revisar su calidad** con `isnull()` y `duplicated()`; **consultar información específica** con `loc[]` y filtros lógicos; y **resumirla en indicadores** con `count()`, `sum()`, `mean()` y `describe()`. Siguiendo ese proceso se identificó que la mayoría de los pedidos se hacen de día y sobre todo en la tarde, que el domingo y el lunes son los días más activos, que los clientes vuelven a comprar con frecuencia (la mitad de los pedidos llega 7 días o menos después del anterior) y que más de la mitad de lo que compran son productos que ya habían pedido antes.

---

### Referencias

Instacart. (2017). *The Instacart Online Grocery Shopping Dataset 2017* [Conjunto de datos]. Kaggle. https://www.kaggle.com/c/instacart-market-basket-analysis

McKinney, W. (2022). *Python for data analysis: Data wrangling with pandas, NumPy, and Jupyter* (3.ª ed.). O'Reilly Media.

Navia Torres, C. E. (2025). *Materiales de ciencia de datos* [Material de formación]. Servicio Nacional de Aprendizaje (SENA). https://educated-show-144.notion.site/Materiales-de-ciencia-de-datos-3fe4671e02a1807aa6cae213e826c33b

The pandas development team. (2026). *pandas documentation*. https://pandas.pydata.org/docs/